# P7: Census Population Merge

Primary merge per `MERGE.md` (§2, **P7**). Concatenates the two cleaned Census
PEP population tables into one long series at **1 row per `place` + `year` +
`estimate_type`**, de-duplicating the overlap year **2020**.

**Inputs** (both `data/tabular/02_clean/census/...`, identical schema):
- `iowa-census-population-2010-2020-clean.csv` (11,304 rows) — intercensal
  vintage. For 2020 it carries only the `census` estimate (the decennial count).
- `iowa-census-population-2020-2025-clean.csv` (6,573 rows) — vintage-2020s. For
  2020 it carries the `estimates_base` (April 1, 2020 anchor) and `july_estimate`.

**Overlap handling.** Only year 2020 appears in both files, but their
`estimate_type` values there are **disjoint** (`census` in the first,
`estimates_base`/`july_estimate` in the second), so a straight concat already
yields a unique key. The de-dup below is therefore a defensive guard: on any
`(place, year, estimate_type)` collision it keeps the **vintage-2020s** row,
since that workbook is the authoritative anchor for the 2020s series.

**Terminal output.** Per `MERGE.md` §6, census population has no join path to
the rest of the pipeline (its key is a city name, not `county_fips` or a station
id, and no city crosswalk exists in `02_clean`). `P7` is produced but **not**
wired into `S2`/`T1`; it is held here for future use.

In [1]:
import os

import pandas as pd

CENSUS = "../../data/tabular/02_clean/census"
OUT_DIR = "../../data/03a_merge_primary"
OUT_FILE = f"{OUT_DIR}/census-population.csv"

KEY = ["place", "year", "estimate_type"]
SCHEMA = ["place", "place_type", "year", "estimate_type", "population"]

## Step 1: Read both vintages

Both files share the same five-column schema. A `source_vintage` provenance
column records which workbook each row came from (`2010_2020` vs `2020_2025`) so
the overlap-year lineage stays traceable after concatenation.

In [2]:
older = pd.read_csv(f"{CENSUS}/iowa-census-population-2010-2020-clean.csv")
newer = pd.read_csv(f"{CENSUS}/iowa-census-population-2020-2025-clean.csv")

assert list(older.columns) == SCHEMA, f"unexpected 2010-2020 schema: {list(older.columns)}"
assert list(newer.columns) == SCHEMA, f"unexpected 2020-2025 schema: {list(newer.columns)}"

older["source_vintage"] = "2010_2020"
newer["source_vintage"] = "2020_2025"

print(f"2010-2020: {len(older):,} rows, years {older['year'].min()}-{older['year'].max()}")
print(f"2020-2025: {len(newer):,} rows, years {newer['year'].min()}-{newer['year'].max()}")
print(f"overlap years: {sorted(set(older['year']) & set(newer['year']))}")

2010-2020: 11,304 rows, years 2010-2020
2020-2025: 6,573 rows, years 2020-2025
overlap years: [2020]


## Step 2: Concatenate and de-duplicate the overlap

The two frames are stacked with `newer` first so that `drop_duplicates(keep="first")`
resolves any `(place, year, estimate_type)` collision in favour of the
vintage-2020s row. With the current data the estimate types are disjoint at 2020,
so no rows are actually dropped — the assertion prints the count for confirmation.

In [3]:
stacked = pd.concat([newer, older], ignore_index=True)
df = stacked.drop_duplicates(subset=KEY, keep="first").copy()

dropped = len(stacked) - len(df)
print(f"stacked: {len(stacked):,} rows | after de-dup: {len(df):,} rows | dropped: {dropped}")
assert not df.duplicated(subset=KEY).any(), "output grain violated: duplicate (place, year, estimate_type)"

df = df.sort_values(KEY).reset_index(drop=True)
print(f"places: {df['place'].nunique():,} | years {df['year'].min()}-{df['year'].max()}")
print(df['estimate_type'].value_counts().to_string())

stacked: 17,877 rows | after de-dup: 17,877 rows | dropped: 0
places: 943 | years 2010-2025
estimate_type
july_estimate     15054
estimates_base     1881
census              942


## Step 3: Save

Columns are ordered as the documented schema (`place`, `place_type`, `year`,
`estimate_type`, `population`) followed by the `source_vintage` provenance column.

In [4]:
df = df[SCHEMA + ["source_vintage"]]

os.makedirs(OUT_DIR, exist_ok=True)
df.to_csv(OUT_FILE, index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} cols -> {OUT_FILE}")
df.head(3)

Saved 17,877 rows x 6 cols -> ../../data/03a_merge_primary/census-population.csv


,place,place_type,year,estimate_type,population,source_vintage
0,Ackley,city,2010,estimates_base,1587,2010_2020
1,Ackley,city,2010,july_estimate,1589,2010_2020
2,Ackley,city,2011,july_estimate,1584,2010_2020
